# Fuzzy Matching addresses to get matching records from HPD and NYCHA violation data

## Making the NYCHA address column

In [1]:
import pandas as pd

In [3]:
nycha_df = pd.read_csv("data/nycha.csv")
hpd_df = pd.read_parquet("data/hpd_violations.parquet")

### Address format:

Primary House Number, Primary Street Name, Primary Borough Name, Primary Postcode (No records have null or blank values for these columns)

Coordinates will be used for extra spatial confirmation later:

Longitude, Latitude

In [7]:
nycha_df['Full_Address'] = (
    nycha_df['Primary House Number'].astype(str) + " " +
    nycha_df['Primary Street Name'].astype(str) + " " + 
    nycha_df['Primary Borough Name'].astype(str) + " " +
    nycha_df['Primary Postcode'].astype(str)
)

In [84]:
nycha_df.tail()

,Violation ID,Primary Building Identifier,Primary Boro Identifier,Primary Borough Name,Primary House Number,Primary Low House Number,Primary High House Number,Primary Street Name,Primary Postcode,Development Name,...,Issued in Error,Latitude,Longitude,Community Board,Council District,BIN,BBL,Census Tract (2020),Neighborhood Tabulation Area (NTA) (2020),Full_Address
2386,18691906,807796,3,BROOKLYN,422,422,430,BLAKE AVENUE,11212,VAN DYKE I,...,N,40.666473,-73.903929,316,41,3328040.0,3.037770e+09,910,BK1602,422 BLAKE AVENUE BROOKLYN 11212
2387,18691907,807796,3,BROOKLYN,422,422,430,BLAKE AVENUE,11212,VAN DYKE I,...,N,40.666473,-73.903929,316,41,3328040.0,3.037770e+09,910,BK1602,422 BLAKE AVENUE BROOKLYN 11212
2388,18692344,989840,1,MANHATTAN,505,505,513,EAST 120 STREET,10035,WAGNER,...,N,40.797582,-73.931095,111,8,1081286.0,1.018080e+09,192,MN1102,505 EAST 120 STREET MANHATTAN 10035
2389,18701333,807796,3,BROOKLYN,422,422,430,BLAKE AVENUE,11212,VAN DYKE I,...,N,40.666473,-73.903929,316,41,3328040.0,3.037770e+09,910,BK1602,422 BLAKE AVENUE BROOKLYN 11212
2390,18701334,807796,3,BROOKLYN,422,422,430,BLAKE AVENUE,11212,VAN DYKE I,...,N,40.666473,-73.903929,316,41,3328040.0,3.037770e+09,910,BK1602,422 BLAKE AVENUE BROOKLYN 11212


### Address columns for HPD to get the same format as above

HouseNumber StreetName Borough Postcode

Coordinates will be used for extra spatial confirmation later:

Longitude, Latitude


### Before creating full_address column for HPD here are the resultsof data integrity checks for these columns ^

#### Nulls?

7524 null Postcodes

The remaining columns have valid values 

In [48]:
# Fill the null postal codes  so they don't break concatenation
hpd_df['Postcode'] = hpd_df['Postcode'].fillna("").astype(str).str.strip()

In [53]:
hpd_df['Full_Address'] = (
    hpd_df['HouseNumber'].astype(str) + " " +
    hpd_df['StreetName'].astype(str) + " " + 
    hpd_df['Borough'].astype(str) + " " +
    hpd_df['Postcode'].astype(str)
)

In [77]:
# remove the trailing .0 at the float -> str in postcode
hpd_df['Full_Address'] = hpd_df['Full_Address'].str.replace(r'\.0$', '', regex=True)
hpd_df['Borough'].unique()

array(['BROOKLYN', 'MANHATTAN', 'BRONX', 'QUEENS', 'STATEN ISLAND'],
      dtype=object)

### Create a new DF where each record is a address fuzzy matched across both data sets
Includes column for NYCHA violation IDs and HPD violation IDs

Since the HPD parquet is so big - split them into DFs by borough first

In [58]:
from rapidfuzz import fuzz, process

In [ ]:
hpd_brooklyn = hpd_df[hpd_df['Borough'] == "BROOKLYN"]
hpd_manhattan = hpd_df[hpd_df['Borough'] == "MANHATTAN"]
hpd_queens = hpd_df[hpd_df['Borough'] == "QUEENS"]
hpd_bronx = hpd_df[hpd_df['Borough'] == "BRONX"]
hpd_si = hpd_df[hpd_df['Borough'] == "STATEN ISLAND"]



In [85]:
borough_map = {
    "BROOKLYN": hpd_brooklyn['Full_Address'].tolist(),
    "BRONX": hpd_bronx['Full_Address'].tolist(),
    "MANHATTAN": hpd_manhattan['Full_Address'].tolist(),
    "QUEENS": hpd_queens['Full_Address'].tolist(),
    "STATEN ISLAND": hpd_si['Full_Address'].tolist()
}

In [87]:
# For each NYCHA address, find best HPD match

results = []

def calc_fuzzy_match(query, hpd_addresses):
    match, score, hpd_index = process.extractOne(query, hpd_addresses)
    return match, score, hpd_index


for idx, row in nycha_df.iterrows():
    borough_name = row['Primary Borough Name']
    hpd_addresses = borough_map.get(borough_name)

    match, score, hpd_index = calc_fuzzy_match(row['Full_Address'], hpd_addresses)
    results.append({
        'nycha_idx': idx,
        'hpd_inx': hpd_index,
        'nycha_address': row['Full_Address'],
        'hpd_address': match,
        'score': score

    })


In [88]:
results_df = pd.DataFrame(results)

total = len(results_df)
high_confidence = results_df[results_df['score'] >= 90]
medium_confidence = results_df[(results_df['score'] >= 70) & (results_df['score'] < 90)]
low_confidence = results_df[results_df['score'] < 70]

print(f"Total matched:       {total}")
print(f"High (>=90):         {len(high_confidence)} ({len(high_confidence)/total*100:.1f}%)")
print(f"Medium (70-89):      {len(medium_confidence)} ({len(medium_confidence)/total*100:.1f}%)")
print(f"Low (<70):           {len(low_confidence)} ({len(low_confidence)/total*100:.1f}%)")

Total matched:       2391
High (>=90):         2305 (96.4%)
Medium (70-89):      86 (3.6%)
Low (<70):           0 (0.0%)


In [94]:
high_confidence.tail()

,nycha_idx,hpd_inx,nycha_address,hpd_address,score
2386,2386,236959,422 BLAKE AVENUE BROOKLYN 11212,442 BLAKE AVENUE BROOKLYN 11212,96.774194
2387,2387,236959,422 BLAKE AVENUE BROOKLYN 11212,442 BLAKE AVENUE BROOKLYN 11212,96.774194
2388,2388,14704,505 EAST 120 STREET MANHATTAN 10035,205 EAST 120 STREET MANHATTAN 10035,97.142857
2389,2389,236959,422 BLAKE AVENUE BROOKLYN 11212,442 BLAKE AVENUE BROOKLYN 11212,96.774194
2390,2390,236959,422 BLAKE AVENUE BROOKLYN 11212,442 BLAKE AVENUE BROOKLYN 11212,96.774194


In [101]:

#medium confidence ones do not look good so here we're just saving high confidence ones
high_confidence.to_csv('initial_address_matches.csv', index=False)
high_confidence.shape

(217, 5)

In [99]:
high_confidence = high_confidence.drop_duplicates(subset=['nycha_address', 'hpd_address'])

A cleaned high_confidence DF with no duplicates has 217 records. This is only 10 addresses less than the original nycha_df number of unique addresses.
Only 10 addresses will not enjoy historic data from the HPD dataset

In [100]:
high_confidence.shape

(217, 5)

In [103]:
len(nycha_df['Full_Address'].unique())

227

In [ ]:
high_confidence.to_csv("cleaned_fuzzy_matching.csv")